# Joining tables with `merge`

`DataFrame.merge` attaches columns from a second table onto the first.
`how` decides which rows stay. The default is `"inner"`.

| Join | Rows that stay |
|---|---|
| inner | keys in both tables |
| left | every row from the left table |
| right | every row from the right table |
| outer | keys from either table |
| cross | every left row paired with every right row |
| semi | left rows whose key is in the right table, left columns only |
| anti | left rows whose key is missing from the right table |

Pandas has no `how="semi"`. We will do a workaround for that.

## How to work through it

Run the cells from the top down. Example cells are filled in.
Each join ends with one medium exercise. Write code in the empty cell.
When the question asks why, answer in a comment.

Put your finished notebook in `week-(current week)/student_work/`.


---
## 1. The two tables

The shared column is `customer_id`.

- Customer 101 has one order.
- Customer 102 has two orders, coffee and biscuit.
- Customer 104 ordered cake, and is missing from `customers`.
- Customer 103 lives in Turku and has no order.


In [1]:
import pandas as pd

# %pip install pandas

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "customer_id": [101, 102, 104, 102],
    "item": ["tea", "coffee", "cake", "biscuit"],
})

customers = pd.DataFrame({
    "customer_id": [101, 102, 103],
    "city": ["Helsinki", "Tampere", "Turku"],
})

display(orders)
display(customers)


,order_id,customer_id,item
0,1,101,tea
1,2,102,coffee
2,3,104,cake
3,4,102,biscuit


,customer_id,city
0,101,Helsinki
1,102,Tampere
2,103,Turku


---
## 2. Inner join

An inner join keeps a row when `customer_id` is in both tables.
Order 3 drops out, because 104 is not in `customers`.
Customer 102 has two orders, so Tampere is attached to both.


In [2]:
orders.merge(customers, on="customer_id", how="inner")


,order_id,customer_id,item,city
0,1,101,tea,Helsinki
1,2,102,coffee,Tampere
2,4,102,biscuit,Tampere


### Exercise 2

Which `order_id` values are in the result?
Which order disappears, and why?


#### ANSWERS:
- on = 'customer_id', how = 'inner':
- 101, 102 are in the table: customer_id in both tables
- 103 and 104 are missing, because they are in either table.

---
## 3. Left join

A left join keeps every row of `orders`.
The city is filled in when that customer exists.
The cake order stays, and its city is missing.


In [3]:
orders.merge(customers, on="customer_id", how="left")

,order_id,customer_id,item,city
0,1,101,tea,Helsinki
1,2,102,coffee,Tampere
2,3,104,cake,NaN
3,4,102,biscuit,Tampere


#### ANSWERS:
- on = 'customer_id', how = 'left':
- 101, 102 and 104 are in the table, because they are in the LEFT table
- 103 is missing, because it is in the right table.

### Exercise 3 

Which item has a missing city? Is that order kept?


#### ANSWERS:
- 104 has a missing city (NaN) because 104 has no value for City in the RIGHT table.

---
## 4. Right join

A right join keeps every row of `customers`.
Turku stays, with no order. The cake order is dropped.
`order_id` prints as `1.0` where an order exists, and `NaN` for Turku.


In [ ]:
orders.merge(customers, on="customer_id", how="right")

,order_id,customer_id,item,city
0,1.0,101,tea,Helsinki
1,2.0,102,coffee,Tampere
2,4.0,102,biscuit,Tampere
3,NaN,103,NaN,Turku


### Exercise 4

Which city has a missing item? Why is the cake order absent?


#### ANSWERS:
- on = 'customer_id', how = 'right':
- Turku has missing items (order_id and item), because Turku city is in the right table but has no value for order_id or item in the LEFT table.

---
## 5. Outer join

An outer join keeps a row when the key is in `orders`, in `customers`, or in both.
Cake stays, with a missing city. Turku stays, with a missing order.
In pandas 3 the unmatched rows sit at the end.


In [6]:
orders.merge(customers, on="customer_id", how="outer")

,order_id,customer_id,item,city
0,1.0,101,tea,Helsinki
1,2.0,102,coffee,Tampere
2,4.0,102,biscuit,Tampere
3,NaN,103,NaN,Turku
4,3.0,104,cake,NaN


### Exercise 5

How many rows are in the result?
Which `customer_id` has no city, and which `customer_id` has no item?


#### ANSWERS:
- on = 'customer_id', how = 'outer':
- There 5 rows in the result.
- customer_id 104 has no city because there is no 'city' value for 104 in any table.
- customer_id 103 has no item because there is no 'item' value for 103 in any table.

---
## 6. Cross join

A cross join pairs every order with every customer. It takes no key.
Both tables have a column named `customer_id`, so the result calls them
`customer_id_x` and `customer_id_y`.
Four orders and three customers give 4 × 3 = 12 rows.


In [7]:
orders.merge(customers, how="cross")


,order_id,customer_id_x,item,customer_id_y,city
0,1,101,tea,101,Helsinki
1,1,101,tea,102,Tampere
2,1,101,tea,103,Turku
3,2,102,coffee,101,Helsinki
4,2,102,coffee,102,Tampere
5,2,102,coffee,103,Turku
6,3,104,cake,101,Helsinki
7,3,104,cake,102,Tampere
8,3,104,cake,103,Turku
9,4,102,biscuit,101,Helsinki


### Exercise 6

How many rows does the cross join return?
Why are there two customer-id columns?


#### ANSWERS:
- orders.merge(customers, how="cross")
- There 12 rows in the cross-join result.
- There are 2 customer_id Columns because they are both appear in 2 tables with the same column name.

---
## 7. Semi join

A semi join keeps left rows whose key is in the right table, and it keeps only the left columns.
Pass the right table as the key column alone, then inner-merge.
The matching orders stay. The `city` column does not come along.


In [8]:
orders.merge(customers[["customer_id"]], on="customer_id", how="inner")

,order_id,customer_id,item
0,1,101,tea
1,2,102,coffee
2,4,102,biscuit


### Exercise 7

Which `order_id` values remain?
Which column from `customers` is absent, and why?


#### ANSWERS:
- on="customer_id", how="semi"
- order_id 101 and 102 values remain because they appear in the right table customer_id column.
- customers column 'city' is absent because they do not appear in the the left table.

---
## 8. Anti join

An anti join keeps left rows whose key is missing from the right table.

In [11]:
display(orders)
display(customers)

,order_id,customer_id,item
0,1,101,tea
1,2,102,coffee
2,3,104,cake
3,4,102,biscuit


,customer_id,city
0,101,Helsinki
1,102,Tampere
2,103,Turku


In [10]:
orders.merge(customers, on="customer_id", how="left_anti", indicator=True )

,order_id,customer_id,item,city,_merge
2,3,104,cake,NaN,left_only


### Exercise 8

What would be the output of `how = "right_anti"` and why? What does the `indicator=True` do?


In [13]:
orders.merge(customers, on="customer_id", how="right_anti", indicator=True )

,order_id,customer_id,item,city,_merge
3,NaN,103,NaN,Turku,right_only


#### ANSWERS:
- on="customer_id", how="right_anti"
- Keep customer_id 103 in the right rows whose key is missing from the Left table
- indicator=True: Create a new column named "_merge" indicating ("right_only" for example) that the values of key are from the right or left table, or both.